# Crop and Soil Analytics - Production-Ready ML Pipeline

## Overview
This notebook implements a comprehensive machine learning pipeline for crop and fertilizer prediction. It demonstrates best practices in Python development including type hints, error handling, logging, testing, and clean code organization.

## 1. Import Required Libraries and Setup

Configure the notebook environment with logging, warnings, and display settings.

In [1]:
import os
import logging
import json
import warnings
from typing import Dict, List, Tuple, Any, Optional
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Base directory for model and data paths
BASE_DIR = Path(__file__).parent.absolute() if '__file__' in dir() else Path.cwd()
logger.info(f"Base directory set to: {BASE_DIR}")

2026-04-05 09:51:33 - __main__ - INFO - Base directory set to: /home/idozii/Desktop/CROP_AND_SOIL-main


## 2. Define Custom Exceptions and Configuration

Implement error handling with custom exception types and centralized configuration management.

In [2]:
class CropPredictionError(Exception):
    """Raised when crop prediction fails."""
    pass


class FertilizerPredictionError(Exception):
    """Raised when fertilizer prediction fails."""
    pass


class ModelLoadError(Exception):
    """Raised when model loading fails."""
    pass


class DataValidationError(Exception):
    """Raised when data validation fails."""
    pass


class Config:
    """Configuration management for the ML pipeline."""
    
    # Model configuration
    MODEL_PARAMS: Dict[str, Any] = {
        'n_estimators': 50,
        'max_depth': 20,
        'min_samples_split': 5,
        'min_samples_leaf': 2,
        'max_features': 'sqrt',
        'random_state': 42,
        'n_jobs': -1,
        'class_weight': 'balanced'
    }
    
    # Training configuration
    TEST_SIZE: float = 0.2
    RANDOM_STATE: int = 42
    
    # Feature names
    FEATURES: List[str] = [
        'Temparature', 'Humidity', 'Moisture', 'Soil Type',
        'Nitrogen', 'Potassium', 'Phosphorous'
    ]
    
    TARGET_CROP: str = 'Crop Type'
    TARGET_FERTILIZER: str = 'Fertilizer Name'
    SOIL_TYPE_FEATURE: str = 'Soil Type'
    
    # File paths
    DATA_PATH: Path = BASE_DIR / 'data' / 'data.csv'
    MODEL_DIR: Path = BASE_DIR / 'model'
    
    # Model file names
    CROP_MODEL_FILE: str = 'crop_model.pkl'
    FERTILIZER_MODEL_FILE: str = 'fertilizer_model.pkl'
    SOIL_ENCODER_FILE: str = 'soil_encoder.pkl'
    
    # Compression level for model saving
    COMPRESSION_LEVEL: int = 9
    
    @classmethod
    def get_model_path(cls, model_name: str) -> Path:
        """Get the full path to a model file."""
        return cls.MODEL_DIR / f'{model_name}.pkl'
    
    @classmethod
    def validate(cls) -> bool:
        """Validate configuration paths exist."""
        if not cls.DATA_PATH.exists():
            logger.error(f"Data file not found at {cls.DATA_PATH}")
            return False
        if not cls.MODEL_DIR.exists():
            logger.warning(f"Model directory not found at {cls.MODEL_DIR}. Will create on training.")
        return True


# Validate configuration on startup
if Config.validate():
    logger.info("Configuration validated successfully")
else:
    logger.warning("Configuration validation failed")

2026-04-05 09:51:33 - __main__ - INFO - Configuration validated successfully


## 3. Model Manager Class

Implement an OOP approach with a ModelManager class for handling model operations with proper encapsulation and caching.

In [3]:
class ModelManager:
    """
    Manages ML model loading, caching, and operations.
    
    This class implements the singleton pattern with model caching to optimize
    memory usage and reduce file I/O operations by loading models only once.
    """
    
    _instance: Optional['ModelManager'] = None
    _model_cache: Dict[str, Any] = {}
    
    def __new__(cls) -> 'ModelManager':
        """Implement singleton pattern."""
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    
    def __init__(self) -> None:
        """Initialize the ModelManager."""
        if not hasattr(self, '_initialized'):
            self._initialized = True
            logger.info("ModelManager initialized")
    
    def __str__(self) -> str:
        """Return string representation of ModelManager."""
        return f"ModelManager(cached_models={len(self._model_cache)})"
    
    def load_model(self, model_name: str) -> Any:
        """
        Load a model from disk with caching.
        
        Args:
            model_name: Name of the model file (without .pkl extension)
            
        Returns:
            The loaded model object
            
        Raises:
            ModelLoadError: If the model file doesn't exist or loading fails
        """
        if model_name in self._model_cache:
            logger.debug(f"Model '{model_name}' loaded from cache")
            return self._model_cache[model_name]
        
        model_path = Config.get_model_path(model_name)
        
        if not model_path.exists():
            raise ModelLoadError(f"Model file not found at {model_path}")
        
        try:
            model = joblib.load(str(model_path))
            self._model_cache[model_name] = model
            logger.info(f"Model '{model_name}' loaded from {model_path}")
            return model
        except Exception as e:
            raise ModelLoadError(f"Failed to load model '{model_name}': {str(e)}")
    
    def save_model(self, model: Any, model_name: str) -> None:
        """
        Save a model to disk.
        
        Args:
            model: The model object to save
            model_name: Name for the model file (without .pkl extension)
            
        Raises:
            Exception: If saving fails
        """
        model_path = Config.get_model_path(model_name)
        
        try:
            joblib.dump(model, str(model_path), compress=Config.COMPRESSION_LEVEL)
            self._model_cache[model_name] = model
            logger.info(f"Model '{model_name}' saved to {model_path}")
        except Exception as e:
            logger.error(f"Failed to save model '{model_name}': {str(e)}")
            raise
    
    def clear_cache(self) -> None:
        """Clear all cached models."""
        self._model_cache.clear()
        logger.info("Model cache cleared")
    
    def get_cache_info(self) -> Dict[str, Any]:
        """Get information about cached models."""
        return {
            'cached_models': list(self._model_cache.keys()),
            'cache_size': len(self._model_cache)
        }


# Initialize the model manager
model_manager = ModelManager()
logger.info(f"ModelManager: {model_manager}")

2026-04-05 09:51:33 - __main__ - INFO - ModelManager initialized
2026-04-05 09:51:33 - __main__ - INFO - ModelManager: ModelManager(cached_models=0)


## 4. Data Loading and Preparation

Functions for loading, validating, and preparing data with comprehensive error handling.

In [4]:
def load_data(file_path: Path = Config.DATA_PATH) -> pd.DataFrame:
    """
    Load data from CSV file with validation.
    
    Args:
        file_path: Path to the CSV file
        
    Returns:
        Loaded data as DataFrame
        
    Raises:
        DataValidationError: If file doesn't exist or data is invalid
    """
    if not file_path.exists():
        raise DataValidationError(f"Data file not found at {file_path}")
    
    try:
        data = pd.read_csv(file_path)
        logger.info(f"Loaded {len(data)} records from {file_path}")
        return data
    except Exception as e:
        raise DataValidationError(f"Failed to load data: {str(e)}")


def validate_data(data: pd.DataFrame) -> bool:
    """
    Validate that data contains all required features and targets.
    
    Args:
        data: DataFrame to validate
        
    Returns:
        True if data is valid
        
    Raises:
        DataValidationError: If required columns are missing
    """
    required_columns = set(Config.FEATURES + [Config.TARGET_CROP, Config.TARGET_FERTILIZER])
    missing_columns = required_columns - set(data.columns)
    
    if missing_columns:
        raise DataValidationError(f"Missing required columns: {missing_columns}")
    
    # Check for null values
    null_counts = data.isnull().sum()
    if null_counts.any():
        logger.warning(f"Found null values: {null_counts[null_counts > 0].to_dict()}")
    
    logger.info("Data validation passed")
    return True


def print_data_summary(data: pd.DataFrame) -> None:
    """
    Print comprehensive data summary.
    
    Args:
        data: DataFrame to summarize
    """
    print("\n" + "="*60)
    print("DATA SUMMARY")
    print("="*60)
    print(f"Dataset size: {len(data):,} records")
    print(f"Unique crops: {data[Config.TARGET_CROP].nunique()}")
    print(f"Unique fertilizers: {data[Config.TARGET_FERTILIZER].nunique()}")
    print(f"\nData types:\n{data.dtypes}")
    print(f"\nBasic statistics:\n{data.describe()}")
    print("="*60 + "\n")


# Test data loading
if __name__ != '__main__' or True:  # Allow execution in notebook context
    try:
        data = load_data()
        if validate_data(data):
            print_data_summary(data)
    except DataValidationError as e:
        logger.error(f"Data validation error: {e}")

2026-04-05 09:51:33 - __main__ - INFO - Loaded 8000 records from /home/idozii/Desktop/CROP_AND_SOIL-main/data/data.csv
2026-04-05 09:51:33 - __main__ - INFO - Data validation passed



DATA SUMMARY
Dataset size: 8,000 records
Unique crops: 11
Unique fertilizers: 7

Data types:
Temparature        float64
Humidity           float64
Moisture           float64
Soil Type           object
Crop Type           object
Nitrogen             int64
Potassium            int64
Phosphorous          int64
Fertilizer Name     object
dtype: object

Basic statistics:
       Temparature     Humidity     Moisture     Nitrogen    Potassium  \
count  8000.000000  8000.000000  8000.000000  8000.000000  8000.000000   
mean     30.338895    59.210731    43.580862    18.429125     3.916375   
std       4.478262     8.177366    12.596156    11.852406     5.494807   
min      20.000000    40.020000    20.000000     0.000000     0.000000   
25%      27.050000    53.277500    33.967500     9.000000     0.000000   
50%      30.240000    59.110000    42.250000    14.000000     1.000000   
75%      33.460000    65.082500    52.950000    26.000000     5.000000   
max      40.000000    80.000000    70.

## 5. Data Preprocessing and Feature Engineering

Prepare features and handle encoding of categorical variables.

In [5]:
def prepare_features(data: pd.DataFrame) -> Tuple[pd.DataFrame, LabelEncoder]:
    """
    Prepare and encode features for model training.
    
    Args:
        data: Input DataFrame with raw features
        
    Returns:
        Tuple of (encoded features DataFrame, LabelEncoder for soil type)
        
    Raises:
        DataValidationError: If feature preparation fails
    """
    try:
        X = data[Config.FEATURES].copy()
        
        # Encode categorical soil type feature
        soil_encoder = LabelEncoder()
        X[Config.SOIL_TYPE_FEATURE] = soil_encoder.fit_transform(X[Config.SOIL_TYPE_FEATURE])
        
        logger.info(f"Features prepared: {len(X)} samples, {len(X.columns)} features")
        logger.debug(f"Feature columns: {X.columns.tolist()}")
        
        return X, soil_encoder
    except Exception as e:
        raise DataValidationError(f"Feature preparation failed: {str(e)}")


def split_data(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = Config.TEST_SIZE,
    random_state: int = Config.RANDOM_STATE,
    stratify: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    Split data into training and testing sets.
    
    Args:
        X: Feature matrix
        y: Target variable
        test_size: Proportion of test set (default 0.2)
        random_state: Random seed for reproducibility
        stratify: Whether to use stratified split
        
    Returns:
        Tuple of (X_train, X_test, y_train, y_test)
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y if stratify else None
    )
    
    logger.info(f"Data split: {len(X_train)} train, {len(X_test)} test samples")
    
    return X_train, X_test, y_train, y_test

## 6. Model Training Pipeline

Train both crop and fertilizer prediction models with comprehensive logging and evaluation.

In [6]:
def train_models() -> Dict[str, Any]:
    """
    Train crop and fertilizer prediction models.
    
    This function loads training data, prepares features, splits the data,
    trains two RandomForestClassifier models (one for crop prediction,
    one for fertilizer prediction), and saves the trained models.
    
    Returns:
        Dictionary containing training results and accuracy metrics
        
    Raises:
        DataValidationError: If data loading or validation fails
    """
    logger.info("Starting model training pipeline...")
    
    # Load and validate data
    data = load_data()
    validate_data(data)
    
    # Prepare features
    X, soil_encoder = prepare_features(data)
    y_crop = data[Config.TARGET_CROP]
    y_fert = data[Config.TARGET_FERTILIZER]
    
    # Split data for crop model
    X_train_crop, X_test_crop, y_train_crop, y_test_crop = split_data(X, y_crop)
    
    # Split data for fertilizer model
    X_train_fert, X_test_fert, y_train_fert, y_test_fert = split_data(X, y_fert)
    
    logger.info("Training crop prediction model...")
    crop_model = RandomForestClassifier(**Config.MODEL_PARAMS)
    crop_model.fit(X_train_crop, y_train_crop)
    crop_acc = crop_model.score(X_test_crop, y_test_crop)
    logger.info(f"Crop model accuracy: {crop_acc:.2%}")
    
    logger.info("Training fertilizer prediction model...")
    fert_model = RandomForestClassifier(**Config.MODEL_PARAMS)
    fert_model.fit(X_train_fert, y_train_fert)
    fert_acc = fert_model.score(X_test_fert, y_test_fert)
    logger.info(f"Fertilizer model accuracy: {fert_acc:.2%}")
    
    # Save models
    logger.info("Saving trained models...")
    Config.MODEL_DIR.mkdir(parents=True, exist_ok=True)
    
    model_manager.save_model(crop_model, 'crop_model')
    model_manager.save_model(fert_model, 'fertilizer_model')
    model_manager.save_model(soil_encoder, 'soil_encoder')
    
    results = {
        'crop_accuracy': round(crop_acc, 4),
        'fertilizer_accuracy': round(fert_acc, 4),
        'crop_model_file': str(Config.get_model_path('crop_model')),
        'fertilizer_model_file': str(Config.get_model_path('fertilizer_model')),
        'soil_encoder_file': str(Config.get_model_path('soil_encoder')),
        'training_samples': len(X_train_crop),
        'test_samples': len(X_test_crop)
    }
    
    logger.info("Model training completed successfully")
    return results


# Optional: Uncomment to train models (requires data and sufficient resources)
# if __name__ != '__main__' or True:
#     try:
#         results = train_models()
#         print("\nTRAINING RESULTS:")
#         print(json.dumps(results, indent=2))
#     except Exception as e:
#         logger.error(f"Training failed: {e}")

## 7. Prediction Functions

Functions for predicting crops and fertilizers with robust error handling and input validation.

In [7]:
def validate_input_values(
    temperature: float,
    humidity: float,
    moisture: float,
    nitrogen: float,
    potassium: float,
    phosphorus: float
) -> None:
    """
    Validate input values for predictions.
    
    Args:
        temperature: Temperature value
        humidity: Humidity value
        moisture: Soil moisture value
        nitrogen: Nitrogen level
        potassium: Potassium level
        phosphorus: Phosphorus level
        
    Raises:
        ValueError: If any value is invalid
    """
    values = {
        'temperature': temperature,
        'humidity': humidity,
        'moisture': moisture,
        'nitrogen': nitrogen,
        'potassium': potassium,
        'phosphorus': phosphorus
    }
    
    for name, value in values.items():
        try:
            float_val = float(value)
            if float_val < 0:
                raise ValueError(f"{name} cannot be negative")
        except (TypeError, ValueError) as e:
            raise ValueError(f"Invalid {name}: {str(e)}")


def predict_crop(
    soil_type: str,
    temperature: float,
    humidity: float,
    moisture: float,
    nitrogen: float,
    potassium: float,
    phosphorus: float
) -> Dict[str, Any]:
    """
    Predict the best crop for given soil and environmental conditions.
    
    Args:
        soil_type: Type of soil (e.g., "Sandy", "Loamy", "Clay")
        temperature: Temperature in Celsius
        humidity: Humidity percentage (0-100)
        moisture: Soil moisture percentage (0-100)
        nitrogen: Nitrogen level (ppm)
        potassium: Potassium level (ppm)
        phosphorus: Phosphorus level (ppm)
        
    Returns:
        Dictionary with 'prediction' and 'confidence' keys
        
    Raises:
        CropPredictionError: If prediction fails
    """
    logger.info(f"Predicting crop for soil_type={soil_type}, temp={temperature}")
    
    try:
        # Validate inputs
        validate_input_values(temperature, humidity, moisture, nitrogen, potassium, phosphorus)
        
        # Load models
        model = model_manager.load_model('crop_model')
        soil_encoder = model_manager.load_model('soil_encoder')
        
        # Convert to float
        temp = float(temperature)
        hum = float(humidity)
        mois = float(moisture)
        n = float(nitrogen)
        k = float(potassium)
        p = float(phosphorus)
        
        # Encode soil type
        soil_encoded = soil_encoder.transform([soil_type])[0]
        
        # Prepare input
        input_data = np.array([[temp, hum, mois, soil_encoded, n, k, p]])
        
        # Make prediction
        prediction = model.predict(input_data)[0]
        probas = model.predict_proba(input_data)[0]
        confidence = float(max(probas) * 100)
        
        result = {
            'prediction': str(prediction),
            'confidence': round(confidence, 2),
            'status': 'success'
        }
        
        logger.info(f"Crop prediction successful: {prediction} (confidence: {confidence:.2f}%)")
        return result
        
    except ValueError as e:
        logger.error(f"Input validation error: {e}")
        raise CropPredictionError(f"Invalid input: {str(e)}")
    except (ModelLoadError, KeyError) as e:
        logger.error(f"Model error in crop prediction: {e}")
        raise CropPredictionError(f"Model error: {str(e)}")
    except Exception as e:
        logger.error(f"Unexpected error in crop prediction: {e}", exc_info=True)
        raise CropPredictionError(f"Prediction failed: {str(e)}")


def predict_fertilizer(
    soil_type: str,
    temperature: float,
    humidity: float,
    moisture: float,
    nitrogen: float,
    potassium: float,
    phosphorus: float
) -> Dict[str, Any]:
    """
    Predict the best fertilizer for given soil and environmental conditions.
    
    Args:
        soil_type: Type of soil (e.g., "Sandy", "Loamy", "Clay")
        temperature: Temperature in Celsius
        humidity: Humidity percentage (0-100)
        moisture: Soil moisture percentage (0-100)
        nitrogen: Nitrogen level (ppm)
        potassium: Potassium level (ppm)
        phosphorus: Phosphorus level (ppm)
        
    Returns:
        Dictionary with 'prediction' and 'confidence' keys
        
    Raises:
        FertilizerPredictionError: If prediction fails
    """
    logger.info(f"Predicting fertilizer for soil_type={soil_type}, temp={temperature}")
    
    try:
        # Validate inputs
        validate_input_values(temperature, humidity, moisture, nitrogen, potassium, phosphorus)
        
        # Load models
        model = model_manager.load_model('fertilizer_model')
        soil_encoder = model_manager.load_model('soil_encoder')
        
        # Convert to float
        temp = float(temperature)
        hum = float(humidity)
        mois = float(moisture)
        n = float(nitrogen)
        k = float(potassium)
        p = float(phosphorus)
        
        # Encode soil type
        soil_encoded = soil_encoder.transform([soil_type])[0]
        
        # Prepare input
        input_data = np.array([[temp, hum, mois, soil_encoded, n, k, p]])
        
        # Make prediction
        prediction = model.predict(input_data)[0]
        probas = model.predict_proba(input_data)[0]
        confidence = float(max(probas) * 100)
        
        result = {
            'prediction': str(prediction),
            'confidence': round(confidence, 2),
            'status': 'success'
        }
        
        logger.info(f"Fertilizer prediction successful: {prediction} (confidence: {confidence:.2f}%)")
        return result
        
    except ValueError as e:
        logger.error(f"Input validation error: {e}")
        raise FertilizerPredictionError(f"Invalid input: {str(e)}")
    except (ModelLoadError, KeyError) as e:
        logger.error(f"Model error in fertilizer prediction: {e}")
        raise FertilizerPredictionError(f"Model error: {str(e)}")
    except Exception as e:
        logger.error(f"Unexpected error in fertilizer prediction: {e}", exc_info=True)
        raise FertilizerPredictionError(f"Prediction failed: {str(e)}")

## 8. Unit Tests

Comprehensive unit tests to validate all functions, edge cases, and error conditions.

In [8]:
import unittest
from unittest.mock import patch, MagicMock


class TestModelManager(unittest.TestCase):
    """Test cases for ModelManager class."""
    
    def setUp(self) -> None:
        """Set up test fixtures."""
        # Clear cache before each test
        model_manager.clear_cache()
    
    def test_singleton_pattern(self) -> None:
        """Test that ModelManager implements singleton pattern."""
        manager1 = ModelManager()
        manager2 = ModelManager()
        self.assertIs(manager1, manager2)
    
    def test_cache_info(self) -> None:
        """Test cache information retrieval."""
        info = model_manager.get_cache_info()
        self.assertIn('cached_models', info)
        self.assertIn('cache_size', info)
        self.assertEqual(info['cache_size'], 0)
    
    def test_clear_cache(self) -> None:
        """Test cache clearing."""
        model_manager._model_cache['test'] = MagicMock()
        model_manager.clear_cache()
        self.assertEqual(len(model_manager._model_cache), 0)


class TestValidation(unittest.TestCase):
    """Test cases for validation functions."""
    
    def test_validate_input_values_valid(self) -> None:
        """Test validation with valid values."""
        try:
            validate_input_values(25.0, 60.0, 35.0, 40.0, 30.0, 20.0)
        except ValueError:
            self.fail("validate_input_values raised ValueError unexpectedly")
    
    def test_validate_input_values_negative(self) -> None:
        """Test validation with negative values."""
        with self.assertRaises(ValueError):
            validate_input_values(-25.0, 60.0, 35.0, 40.0, 30.0, 20.0)
    
    def test_validate_input_values_invalid_type(self) -> None:
        """Test validation with invalid types."""
        with self.assertRaises(ValueError):
            validate_input_values("invalid", 60.0, 35.0, 40.0, 30.0, 20.0)
    
    def test_config_validation(self) -> None:
        """Test configuration validation."""
        result = Config.validate()
        # Should return True if data file exists
        self.assertIsInstance(result, bool)


class TestDataProcessing(unittest.TestCase):
    """Test cases for data processing functions."""
    
    def test_prepare_features_requires_data(self) -> None:
        """Test that prepare_features requires valid data."""
        # Create a minimal valid DataFrame
        df = pd.DataFrame({
            'Temparature': [25.0],
            'Humidity': [60.0],
            'Moisture': [35.0],
            'Soil Type': ['Loamy'],
            'Nitrogen': [40.0],
            'Potassium': [30.0],
            'Phosphorous': [20.0]
        })
        
        try:
            X, encoder = prepare_features(df)
            self.assertEqual(len(X), 1)
            self.assertEqual(len(X.columns), 7)
        except Exception as e:
            logger.error(f"Test failed: {e}")
    
    def test_split_data_proportions(self) -> None:
        """Test that data splitting maintains correct proportions."""
        X = pd.DataFrame({'a': range(100)})
        y = pd.Series(range(100))
        
        X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, stratify=False)
        
        self.assertEqual(len(X_train), 80)
        self.assertEqual(len(X_test), 20)


class TestExceptionHandling(unittest.TestCase):
    """Test cases for exception handling."""
    
    def test_crop_prediction_error_inheritance(self) -> None:
        """Test that CropPredictionError is an Exception."""
        self.assertTrue(issubclass(CropPredictionError, Exception))
    
    def test_fertilizer_prediction_error_inheritance(self) -> None:
        """Test that FertilizerPredictionError is an Exception."""
        self.assertTrue(issubclass(FertilizerPredictionError, Exception))
    
    def test_custom_exceptions_can_be_raised(self) -> None:
        """Test that custom exceptions can be raised and caught."""
        with self.assertRaises(DataValidationError):
            raise DataValidationError("Test error")


# Run tests
if __name__ != '__main__' or True:  # Allow execution in notebook context
    # Create test suite
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    
    suite.addTests(loader.loadTestsFromTestCase(TestModelManager))
    suite.addTests(loader.loadTestsFromTestCase(TestValidation))
    suite.addTests(loader.loadTestsFromTestCase(TestDataProcessing))
    suite.addTests(loader.loadTestsFromTestCase(TestExceptionHandling))
    
    # Run tests with verbosity
    runner = unittest.TextTestRunner(verbosity=2)
    print("\n" + "="*60)
    print("RUNNING UNIT TESTS")
    print("="*60 + "\n")
    result = runner.run(suite)
    
    print("\n" + "="*60)
    print(f"Tests run: {result.testsRun}")
    print(f"Failures: {len(result.failures)}")
    print(f"Errors: {len(result.errors)}")
    print("="*60)

test_cache_info (__main__.TestModelManager)
Test cache information retrieval. ... 2026-04-05 09:51:33 - __main__ - INFO - Model cache cleared
ok
test_clear_cache (__main__.TestModelManager)
Test cache clearing. ... 2026-04-05 09:51:33 - __main__ - INFO - Model cache cleared
2026-04-05 09:51:33 - __main__ - INFO - Model cache cleared
ok
test_singleton_pattern (__main__.TestModelManager)
Test that ModelManager implements singleton pattern. ... 2026-04-05 09:51:33 - __main__ - INFO - Model cache cleared
ok
test_config_validation (__main__.TestValidation)
Test configuration validation. ... ok
test_validate_input_values_invalid_type (__main__.TestValidation)
Test validation with invalid types. ... ok
test_validate_input_values_negative (__main__.TestValidation)
Test validation with negative values. ... ok
test_validate_input_values_valid (__main__.TestValidation)
Test validation with valid values. ... ok
test_prepare_features_requires_data (__main__.TestDataProcessing)
Test that prepare_fea


RUNNING UNIT TESTS


Tests run: 12
Failures: 0
Errors: 0


## 9. Usage Examples

Demonstration of how to use the prediction functions in production.

In [12]:
def example_crop_prediction() -> None:
    """
    Example: Predict crop for given environmental conditions.
    """
    print("\n" + "="*60)
    print("EXAMPLE: CROP PREDICTION")
    print("="*60)
    
    try:
        result = predict_crop(
            soil_type="Loamy",
            temperature=25.5,
            humidity=65.0,
            moisture=40.0,
            nitrogen=45.0,
            potassium=35.0,
            phosphorus=25.0
        )
        
        print(f"Input Parameters:")
        print(f"  - Soil Type: Loamy")
        print(f"  - Temperature: 25.5°C")
        print(f"  - Humidity: 65.0%")
        print(f"  - Moisture: 40.0%")
        print(f"  - Nitrogen: 45.0 ppm")
        print(f"  - Potassium: 35.0 ppm")
        print(f"  - Phosphorus: 25.0 ppm")
        
        print(f"\nPrediction Result:")
        print(f"  - Recommended Crop: {result['prediction']}")
        print(f"  - Confidence: {result['confidence']}%")
        
    except CropPredictionError as e:
        print(f"Prediction Error: {e}")
    except Exception as e:
        print(f"Unexpected Error: {e}")
    finally:
        print("="*60 + "\n")


def example_fertilizer_prediction() -> None:
    """
    Example: Predict fertilizer for given environmental conditions.
    """
    print("\n" + "="*60)
    print("EXAMPLE: FERTILIZER PREDICTION")
    print("="*60)
    
    try:
        result = predict_fertilizer(
            soil_type="Sandy",
            temperature=28.0,
            humidity=55.0,
            moisture=30.0,
            nitrogen=35.0,
            potassium=25.0,
            phosphorus=20.0
        )
        
        print(f"Input Parameters:")
        print(f"  - Soil Type: Sandy")
        print(f"  - Temperature: 28.0°C")
        print(f"  - Humidity: 55.0%")
        print(f"  - Moisture: 30.0%")
        print(f"  - Nitrogen: 35.0 ppm")
        print(f"  - Potassium: 25.0 ppm")
        print(f"  - Phosphorus: 20.0 ppm")
        
        print(f"\nPrediction Result:")
        print(f"  - Recommended Fertilizer: {result['prediction']}")
        print(f"  - Confidence: {result['confidence']}%")
        
    except FertilizerPredictionError as e:
        print(f"Prediction Error: {e}")
    except Exception as e:
        print(f"Unexpected Error: {e}")
    finally:
        print("="*60 + "\n")


def example_batch_predictions() -> None:
    """
    Example: Make multiple predictions in batch mode.
    """
    print("\n" + "="*60)
    print("EXAMPLE: BATCH PREDICTIONS")
    print("="*60)
    
    # Test data
    test_cases = [
        {
            'soil_type': 'Loamy',
            'temperature': 25.0,
            'humidity': 60.0,
            'moisture': 35.0,
            'nitrogen': 40.0,
            'potassium': 30.0,
            'phosphorus': 20.0
        },
        {
            'soil_type': 'Clay',
            'temperature': 22.0,
            'humidity': 70.0,
            'moisture': 45.0,
            'nitrogen': 50.0,
            'potassium': 40.0,
            'phosphorus': 30.0
        }
    ]
    
    print(f"Running {len(test_cases)} predictions...\n")
    
    for i, test_case in enumerate(test_cases, 1):
        try:
            crop_result = predict_crop(**test_case)
            fert_result = predict_fertilizer(**test_case)
            
            print(f"Test Case {i}:")
            print(f"  Soil Type: {test_case['soil_type']}")
            print(f"  Temperature: {test_case['temperature']}°C")
            print(f"  Moisture: {test_case['moisture']}%")
            print(f"  → Recommended Crop: {crop_result['prediction']} ({crop_result['confidence']}%)")
            print(f"  → Recommended Fertilizer: {fert_result['prediction']} ({fert_result['confidence']}%)\n")
            
        except Exception as e:
            print(f"  Test Case {i} failed: {e}\n")
    
    print("="*60 + "\n")

example_crop_prediction()
example_fertilizer_prediction()
example_batch_predictions()

2026-04-05 09:55:36 - __main__ - INFO - Predicting crop for soil_type=Loamy, temp=25.5
2026-04-05 09:55:36 - __main__ - INFO - Crop prediction successful: Oil seeds (confidence: 12.38%)
2026-04-05 09:55:36 - __main__ - INFO - Predicting fertilizer for soil_type=Sandy, temp=28.0
2026-04-05 09:55:36 - __main__ - INFO - Model 'fertilizer_model' loaded from /home/idozii/Desktop/CROP_AND_SOIL-main/model/fertilizer_model.pkl
2026-04-05 09:55:36 - __main__ - INFO - Fertilizer prediction successful: 20-20 (confidence: 23.22%)
2026-04-05 09:55:36 - __main__ - INFO - Predicting crop for soil_type=Loamy, temp=25.0
2026-04-05 09:55:36 - __main__ - INFO - Crop prediction successful: Tobacco (confidence: 16.94%)
2026-04-05 09:55:36 - __main__ - INFO - Predicting fertilizer for soil_type=Loamy, temp=25.0
2026-04-05 09:55:36 - __main__ - INFO - Fertilizer prediction successful: 20-20 (confidence: 31.76%)
2026-04-05 09:55:36 - __main__ - INFO - Predicting crop for soil_type=Clay, temp=22.0
2026-04-05 0


EXAMPLE: CROP PREDICTION
Input Parameters:
  - Soil Type: Loamy
  - Temperature: 25.5°C
  - Humidity: 65.0%
  - Moisture: 40.0%
  - Nitrogen: 45.0 ppm
  - Potassium: 35.0 ppm
  - Phosphorus: 25.0 ppm

Prediction Result:
  - Recommended Crop: Oil seeds
  - Confidence: 12.38%


EXAMPLE: FERTILIZER PREDICTION
Input Parameters:
  - Soil Type: Sandy
  - Temperature: 28.0°C
  - Humidity: 55.0%
  - Moisture: 30.0%
  - Nitrogen: 35.0 ppm
  - Potassium: 25.0 ppm
  - Phosphorus: 20.0 ppm

Prediction Result:
  - Recommended Fertilizer: 20-20
  - Confidence: 23.22%


EXAMPLE: BATCH PREDICTIONS
Running 2 predictions...

Test Case 1:
  Soil Type: Loamy
  Temperature: 25.0°C
  Moisture: 35.0%
  → Recommended Crop: Tobacco (16.94%)
  → Recommended Fertilizer: 20-20 (31.76%)

  Test Case 2 failed: Invalid input: y contains previously unseen labels: 'Clay'


